# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 6.8 MB/s eta 0:00:00
dependencies ok


In [4]:
import re
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference,helper,numpy_helper,TensorProto

In [5]:
TASK_ID = 'task094'
CH = 10
H = W = 30
INPUT_SHAPE = [1, CH, H, W]
OUTPUT_SHAPE = [1, CH, H, W]
FORBIDDEN = {'Loop', 'Scan', 'NonZero', 'Unique', 'Script', 'Function'}

task_candidates = [
    Path('/kaggle/input/competitions/neurogolf-2026/task094.json'),
    Path('/mnt/data/task094.json'),
    Path.cwd() / 'task094.json',
    Path.cwd() / 'recovered' / 'task094.json',
]
TASK_PATH = next((p for p in task_candidates if p.exists()), None)
if TASK_PATH is None:
    raise FileNotFoundError('task094.json was not found')

WORK = Path.cwd()/'working'
WORK.mkdir(parents=True, exist_ok=True)

ONNX_PATH = WORK / 'task094.onnx'
SUBMISSION_PATH = Path.cwd() / 'submission.zip'


SUMMARY_PATH = WORK / 'task094_v25_validation_summary.json'

task = json.loads(TASK_PATH.read_text())
print('task:', TASK_PATH)
print({k: len(task.get(k, [])) for k in ['train', 'test', 'arc-gen']})


task: /kaggle/input/competitions/neurogolf-2026/task094.json
{'train': 2, 'test': 1, 'arc-gen': 262}


In [6]:
def grid_to_tensor(grid, offset=(0, 0)):
    a = np.asarray(grid, dtype=np.int64)
    h, w = a.shape
    r0, c0 = offset
    assert 0 <= r0 <= H - h and 0 <= c0 <= W - w
    x = np.zeros(INPUT_SHAPE, dtype=np.float32)
    rr, cc = np.indices((h, w))
    x[0, a, rr + r0, cc + c0] = 1.0
    return x


def five_run_oracle(grid):
    g = np.asarray(grid, dtype=np.int64)
    h, w = g.shape
    center_rows = set()
    center_cols = set()

    for r in range(h):
        for left in range(max(0, w - 4)):
            if np.all(g[r, left:left + 5] == 1):
                center_cols.add(left + 2)

    for top in range(max(0, h - 4)):
        for c in range(w):
            if np.all(g[top:top + 5, c] == 1):
                center_rows.add(top + 2)

    out = g.copy()
    for r in center_rows:
        out[r, out[r] == 8] = 6
    for c in center_cols:
        mask = out[:, c] == 8
        out[mask, c] = 6
    return out


def exact_raw(y, expected):
    return np.array_equal(y, expected)


In [7]:
def init(name, array, dtype=None):
    a = np.asarray(array, dtype=dtype)
    return numpy_helper.from_array(a, name=name)


initializers = [
    init('idx_blue', [1], np.int64),
    init('idx_cyan', [8], np.int64),
    init('axis_channel', [1], np.int64),
    init('five_threshold', [4.5], np.float32),
    init('active_threshold', [0.5], np.float32),
    init('one', [1.0], np.float32),
    init('expand_shape', [1, 1, H, W], np.int64),
    init('h_kernel', np.ones((1, 1, 1, 5), dtype=np.float32)),
    init('v_kernel', np.ones((1, 1, 5, 1), dtype=np.float32)),
    init('color6', np.eye(CH, dtype=np.float32)[6].reshape(1, CH, 1, 1)),
]

nodes = [
    helper.make_node('Gather', ['input', 'idx_blue'], ['blue'], axis=1, name='blue_channel'),
    helper.make_node('Gather', ['input', 'idx_cyan'], ['cyan'], axis=1, name='cyan_channel'),
    helper.make_node('ReduceSum', ['input', 'axis_channel'], ['active_sum'], keepdims=1, name='active_sum'),
    helper.make_node('Greater', ['active_sum', 'active_threshold'], ['active_bool'], name='active_bool'),
    helper.make_node('Cast', ['active_bool'], ['active'], to=TensorProto.FLOAT, name='active_float'),

    helper.make_node('Conv', ['blue', 'h_kernel'], ['h_sum'], pads=[0, 2, 0, 2], name='horizontal_five_sum'),
    helper.make_node('Greater', ['h_sum', 'five_threshold'], ['h_hit_bool'], name='horizontal_hit_bool'),
    helper.make_node('Cast', ['h_hit_bool'], ['h_hit'], to=TensorProto.FLOAT, name='horizontal_hit'),
    helper.make_node('ReduceMax', ['h_hit'], ['center_columns'], axes=[2], keepdims=1, name='center_columns'),
    helper.make_node('Expand', ['center_columns', 'expand_shape'], ['vertical_axes'], name='vertical_axes'),

    helper.make_node('Conv', ['blue', 'v_kernel'], ['v_sum'], pads=[2, 0, 2, 0], name='vertical_five_sum'),
    helper.make_node('Greater', ['v_sum', 'five_threshold'], ['v_hit_bool'], name='vertical_hit_bool'),
    helper.make_node('Cast', ['v_hit_bool'], ['v_hit'], to=TensorProto.FLOAT, name='vertical_hit'),
    helper.make_node('ReduceMax', ['v_hit'], ['center_rows'], axes=[3], keepdims=1, name='center_rows'),
    helper.make_node('Expand', ['center_rows', 'expand_shape'], ['horizontal_axes'], name='horizontal_axes'),

    helper.make_node('Max', ['vertical_axes', 'horizontal_axes'], ['all_axes'], name='all_axes'),
    helper.make_node('Mul', ['all_axes', 'active'], ['masked_axes'], name='active_canvas_axes'),
    helper.make_node('Mul', ['masked_axes', 'cyan'], ['paint'], name='paint_cyan_only'),
    helper.make_node('Sub', ['one', 'paint'], ['keep_mask'], name='keep_mask'),
    helper.make_node('Mul', ['input', 'keep_mask'], ['kept_input'], name='preserve_input'),
    helper.make_node('Mul', ['color6', 'paint'], ['magenta'], name='magenta_pixels'),
    helper.make_node('Add', ['kept_input', 'magenta'], ['composed'], name='compose'),
    helper.make_node('Mul', ['composed', 'active'], ['output'], name='zero_padding'),
]

graph = helper.make_graph(
    nodes,
    'task094_v25_codegolf_consensus_five_runs',
    [helper.make_tensor_value_info('input', TensorProto.FLOAT, INPUT_SHAPE)],
    [helper.make_tensor_value_info('output', TensorProto.FLOAT, OUTPUT_SHAPE)],
    initializer=initializers,
)
model = helper.make_model(
    graph,
    producer_name='task094-v25',
    opset_imports=[helper.make_opsetid('', 17)],
)
model.ir_version = 10
model = shape_inference.infer_shapes(model)
onnx.checker.check_model(model)
onnx.save(model, ONNX_PATH)
print('wrote', ONNX_PATH, ONNX_PATH.stat().st_size, 'bytes')


wrote /kaggle/working/working/task094.onnx 2691 bytes


In [8]:
model = onnx.load(ONNX_PATH)
ops = collections.Counter(node.op_type for node in model.graph.node)
input_shape = [d.dim_value for d in model.graph.input[0].type.tensor_type.shape.dim]
output_shape = [d.dim_value for d in model.graph.output[0].type.tensor_type.shape.dim]
forbidden = sorted(FORBIDDEN & set(ops))

assert input_shape == INPUT_SHAPE, input_shape
assert output_shape == OUTPUT_SHAPE, output_shape
assert not forbidden, forbidden
assert len(model.functions) == 0
assert ONNX_PATH.stat().st_size < 1_400_000

static_summary = {
    'input_shape': input_shape,
    'output_shape': output_shape,
    'onnx_size_bytes': ONNX_PATH.stat().st_size,
    'node_count': len(model.graph.node),
    'ops': dict(ops),
    'forbidden_ops': forbidden,
    'ir_version': model.ir_version,
    'opset': model.opset_import[0].version,
}
print(json.dumps(static_summary, indent=2))


{
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 2691,
  "node_count": 23,
  "ops": {
    "Gather": 2,
    "ReduceSum": 1,
    "Greater": 3,
    "Cast": 3,
    "Conv": 2,
    "ReduceMax": 2,
    "Expand": 2,
    "Max": 1,
    "Mul": 5,
    "Sub": 1,
    "Add": 1
  },
  "forbidden_ops": [],
  "ir_version": 10,
  "opset": 17
}


In [9]:
session_options = ort.SessionOptions()
session_options.intra_op_num_threads = 1
session_options.inter_op_num_threads = 1
session = ort.InferenceSession(str(ONNX_PATH), session_options, providers=['CPUExecutionProvider'])
assert session.get_inputs()[0].name == 'input'
assert session.get_outputs()[0].name == 'output'


def run_tensor(x):
    return session.run(['output'], {'input': x.astype(np.float32)})[0]


def validate_examples(examples):
    ok = padding_ok = unambiguous_ok = 0
    bad = []
    for i, ex in enumerate(examples):
        g = np.asarray(ex['input'], dtype=np.int64)
        expected = grid_to_tensor(ex['output'])
        y = run_tensor(grid_to_tensor(g))
        if exact_raw(y, expected):
            ok += 1
        else:
            bad.append(i)
        active = grid_to_tensor(g).sum(axis=1, keepdims=True) > 0.5
        padding_ok += int(np.max(np.abs(y * (~active))) == 0.0)
        inside = active[0, 0]
        unambiguous_ok += int(np.all(y[0].sum(axis=0)[inside] == 1.0))
    return {
        'ok': ok,
        'total': len(examples),
        'bad_first10': bad[:10],
        'padding_ok': padding_ok,
        'unambiguous_ok': unambiguous_ok,
    }


official = {}
for split in ['train', 'test', 'arc-gen']:
    official[split] = validate_examples(task.get(split, []))
    print(split, official[split])
    assert official[split]['ok'] == official[split]['total']
    assert official[split]['padding_ok'] == official[split]['total']
    assert official[split]['unambiguous_ok'] == official[split]['total']

zero_y = run_tensor(np.zeros(INPUT_SHAPE, dtype=np.float32))
assert np.array_equal(zero_y, np.zeros(OUTPUT_SHAPE, dtype=np.float32))
print('all-zero profiler input: exact zero output')


train {'ok': 2, 'total': 2, 'bad_first10': [], 'padding_ok': 2, 'unambiguous_ok': 2}
test {'ok': 1, 'total': 1, 'bad_first10': [], 'padding_ok': 1, 'unambiguous_ok': 1}
arc-gen {'ok': 262, 'total': 262, 'bad_first10': [], 'padding_ok': 262, 'unambiguous_ok': 262}
all-zero profiler input: exact zero output


In [10]:
def make_grid(h=15, w=15, cells=(), extras=()):
    g = np.full((h, w), 8, dtype=np.int64)
    for r, c in cells:
        g[r, c] = 1
    for r, c, color in extras:
        g[r, c] = color
    return g


stress = []

def add_case(name, grid):
    stress.append({'name': name, 'input': grid.tolist(), 'output': five_run_oracle(grid).tolist()})


for length in range(1, 9):
    add_case(f'horizontal_length_{length}', make_grid(cells={(7, c) for c in range(3, 3 + length)}))
    add_case(f'vertical_length_{length}', make_grid(cells={(r, 7) for r in range(3, 3 + length)}))

add_case('two_horizontal_bars', make_grid(cells={(4, c) for c in range(2, 7)} | {(10, c) for c in range(8, 13)}))
add_case('two_vertical_bars', make_grid(cells={(r, 4) for r in range(2, 7)} | {(r, 10) for r in range(8, 13)}))
add_case('cross_of_bars', make_grid(cells={(7, c) for c in range(5, 10)} | {(r, 7) for r in range(5, 10)}))
add_case('open_frame_top_bottom', make_grid(cells={(5, c) for c in range(5, 10)} | {(9, c) for c in range(5, 10)}))
add_case('open_frame_left_right', make_grid(cells={(r, 5) for r in range(5, 10)} | {(r, 9) for r in range(5, 10)}))
add_case('distractors', make_grid(cells={(7, c) for c in range(5, 10)}, extras=[(1, 1, 3), (13, 13, 9)]))
add_case('rectangular_canvas', make_grid(11, 19, cells={(5, c) for c in range(7, 12)} | {(r, 15) for r in range(3, 8)}))

rng = random.Random(25094)
for i in range(200):
    h = rng.randint(7, 30)
    w = rng.randint(7, 30)
    cells = set()
    for _ in range(rng.randint(1, 5)):
        if rng.random() < 0.5:
            r = rng.randrange(h)
            left = rng.randrange(w - 4)
            cells.update((r, c) for c in range(left, left + rng.randint(5, min(8, w - left))))
        else:
            top = rng.randrange(h - 4)
            c = rng.randrange(w)
            cells.update((r, c) for r in range(top, top + rng.randint(5, min(8, h - top))))
    add_case(f'random_{i:03d}', make_grid(h, w, cells=cells))

stress_result = validate_examples(stress)
print('stress', stress_result)
assert stress_result['ok'] == stress_result['total']
assert stress_result['padding_ok'] == stress_result['total']
assert stress_result['unambiguous_ok'] == stress_result['total']

shifted_ok = 0
for case, (r0, c0) in zip(stress[:12], [(1, 2), (3, 4), (5, 1), (2, 6), (4, 3), (1, 7), (6, 2), (3, 5), (2, 1), (1, 3), (4, 2), (2, 4)]):
    gi = np.asarray(case['input'])
    go = np.asarray(case['output'])
    if r0 + gi.shape[0] > H or c0 + gi.shape[1] > W:
        continue
    x = grid_to_tensor(gi, (r0, c0))
    expected = grid_to_tensor(go, (r0, c0))
    shifted_ok += int(exact_raw(run_tensor(x), expected))
print('shifted active canvases:', shifted_ok)
assert shifted_ok > 0


stress {'ok': 223, 'total': 223, 'bad_first10': [], 'padding_ok': 223, 'unambiguous_ok': 223}
shifted active canvases: 12


In [11]:


def finalist_one(grid):
    # Clean equivalent of the published 2025 first-place task094 program.
    g = np.asarray(grid, dtype=np.int64).tolist()
    pattern = r'8(?=(.{47})*, 1, 1, [86])'
    for _ in range(4):
        painted = eval(re.sub(pattern, '6', str(g)))
        g = [list(row) for row in [*zip(*painted)][::-1]]
    return np.asarray(g, dtype=np.int64)


def finalist_two(grid):
    # Clean equivalent of the published 2025 second-place task094 program.
    pattern = r'8(?=[^(]*+[^)]*1.{46}1, 1)'
    g = np.asarray(grid, dtype=np.int64).tolist()
    for _ in range(2):
        transposed = tuple(zip(*g))
        g = [list(row) for row in eval(re.sub(pattern, '6', f'{*transposed,}'))]
    return np.asarray(g, dtype=np.int64)


consensus_rng = random.Random(41_17)
consensus_total = 0
consensus_matched = 0
for _ in range(1200):
    top = consensus_rng.randrange(0, 11)
    left = consensus_rng.randrange(0, 11)
    count = consensus_rng.randrange(0, 26)
    local = consensus_rng.sample([(r, c) for r in range(5) for c in range(5)], count)
    cells = {(top + r, left + c) for r, c in local}
    grid = make_grid(cells=cells)
    a = finalist_one(grid)
    b = finalist_two(grid)
    if np.array_equal(a, b):
        consensus_total += 1
        consensus_matched += int(np.array_equal(a, five_run_oracle(grid)))

for r in range(3, 12):
    for c in range(3, 12):
        ring = {(r - 2, j) for j in range(c - 2, c + 3)} | {(r + 2, j) for j in range(c - 2, c + 3)}
        ring |= {(i, c - 2) for i in range(r - 2, r + 3)} | {(i, c + 2) for i in range(r - 2, r + 3)}
        grid = make_grid(cells=ring)
        a = finalist_one(grid)
        b = finalist_two(grid)
        assert np.array_equal(a, b)
        assert np.array_equal(a, five_run_oracle(grid))

finalist_consensus = {'matched': consensus_matched, 'total': consensus_total}
print('published-finalist consensus proxy:', finalist_consensus)
assert consensus_total > 0
assert consensus_matched == consensus_total


published-finalist consensus proxy: {'matched': 181, 'total': 181}


In [12]:
summary = {
    'task_id': TASK_ID,
    'model_tag': 'v25_codegolf_consensus_five_runs',
    'hypothesis': 'five consecutive blue pixels vote for the orthogonal center axis; cyan 8 changes to magenta 6',
    'static': static_summary,
    'official': official,
    'stress': stress_result,
    'shifted_active_canvas_ok': shifted_ok,
    'published_finalist_consensus_proxy': finalist_consensus,
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))

for path in [SUBMISSION_PATH]:
    if path.exists():
        path.unlink()
    with zipfile.ZipFile(path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(ONNX_PATH, arcname='task094.onnx')
    with zipfile.ZipFile(path) as zf:
        assert zf.namelist() == ['task094.onnx']
        assert zf.getinfo('task094.onnx').file_size == ONNX_PATH.stat().st_size

digest = hashlib.sha256(ONNX_PATH.read_bytes()).hexdigest()
print('summary:', SUMMARY_PATH)
print('submission:', SUBMISSION_PATH, SUBMISSION_PATH.stat().st_size, 'bytes')

print('onnx sha256:', digest)
print('READY')


summary: /kaggle/working/working/task094_v25_validation_summary.json
submission: /kaggle/working/submission.zip 1033 bytes
onnx sha256: 27a170f3e6f8addf242f425c1249114ad9400f68930f1f5b06c54a7559e84173
READY
